# Explainability of BERT with SHAP, LIME, and Integrated Gradients #

## Setup and Import Model ##

In [1]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
  print(f'Running on GPU: {torch.cuda.get_device_name(0)}')
else:
  print('Running on CPU')

Running on GPU: NVIDIA GeForce RTX 2080 Super with Max-Q Design


### Importing Dataset ###

In [2]:
import pandas as pd

# path of each of the three csv files of the GoEmotions dataset
# Assuming the files are directly in MyDrive based on the file list
csv_path1 = 'C:/Users/maewa/Coding/IntProjectII/IMCS3020U_Project_TBA/data/goemotions_1.csv'
csv_path2 = 'C:/Users/maewa/Coding/IntProjectII/IMCS3020U_Project_TBA/data/goemotions_2.csv'
csv_path3 = 'C:/Users/maewa/Coding/IntProjectII/IMCS3020U_Project_TBA/data/goemotions_3.csv'

# List of the three files
csv_files = [csv_path1, csv_path2, csv_path3]

# generate DataFrame from each file then concatenate all three DataFrames into one
df = pd.concat((pd.read_csv(filename) for filename in csv_files), ignore_index=True)

#isolate the columns for emotion labels
label_cols = df.columns[9:].tolist()
#print(df.columns)
#print(label_cols)

# turn text column into lists
texts = df['text'].tolist()
# results in a list of lists representing the labels
labels = df[label_cols].values.tolist()

### Load BERT Tokenizer ###

In [3]:
from transformers import AutoTokenizer

MODEL_NAME = 'bert-base-uncased'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'Tokenizer type loaded: {type(tokenizer).__name__}')


c:\Users\maewa\anaconda3\envs\bert_xai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer type loaded: BertTokenizer


### Define model ###

In [4]:
from transformers import AutoModelForSequenceClassification, AutoConfig

NUM_LABELS = 28

# configuration for multilabel classification
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
  print(f"Running on GPU: {torch.cuda.get_device_name(0)}")
else:
  print("Running on CPU (GPU not available or selected)")

model.to(device)

print(f"Model Loaded: {MODEL_NAME} with {NUM_LABELS} output labels.")
print(f"Running on device: {device}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 499.71it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

Running on GPU: NVIDIA GeForce RTX 2080 Super with Max-Q Design
Model Loaded: bert-base-uncased with 28 output labels.
Running on device: cuda
